# B2-020-language-transformers — Practice p17 — Solution

**Type:** integrative · **Difficulty:** advanced · **Concepts:** embedding-model-training, learned-token-embedding

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).

## Pinned reproducibility protocol

Import only the literal pairs and vocabulary IDs from `data/language_fixture.py`.  Seed Python, NumPy, and Torch with `20260812`; initialize a `(12,8)` embedding table and distinct `Linear(8,12)` head from that seed.  Use AdamW with `lr=0.03`, `weight_decay=0`, betas `(0.9,0.999)`, eps `1e-8`, and no shuffle.  Run exactly 40 full-batch updates in stored ascending order, with `zero_grad(set_to_none=True)`, backward, then step.

## Solution

The literal same-token causal pairs supply a small predictive bridge. Row 4's nearest neighbors are reported before and after training; this is learned-vector evidence because predictive gradients move selected rows, but it is not external-corpus pretraining because all pairs come from the checked-in synthetic fixture.

In [ ]:
import importlib.util

def load_literal_module(name, relative_path):
    spec = importlib.util.spec_from_file_location(name, relative_path)
    assert spec is not None and spec.loader is not None
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

import random
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

fixture = load_literal_module("language_fixture_p17", "../data/language_fixture.py")
random.seed(20260812); np.random.seed(20260812); torch.manual_seed(20260812)
table = nn.Parameter(torch.randn(12, 8, dtype=torch.float32))
head = nn.Linear(8, 12, dtype=torch.float32)
context_ids = torch.tensor(fixture.P17_CONTEXT_IDS, dtype=torch.int64)
target_ids = torch.tensor(fixture.P17_TARGET_IDS, dtype=torch.int64)
before = table.detach().clone()

def bridge_loss():
    return F.cross_entropy(head(table[context_ids]), target_ids, reduction="mean")

def nearest_ids(values, row=4, k=3):
    similarities = F.cosine_similarity(values[row].unsqueeze(0), values, dim=1)
    similarities[row] = -2
    return torch.topk(similarities, k).indices.tolist()

initial_loss = bridge_loss().detach()
neighbors_before = nearest_ids(before.clone())
optimizer = torch.optim.AdamW(
    [table, *head.parameters()], lr=0.03, weight_decay=0,
    betas=(0.9, 0.999), eps=1e-8,
)
for _ in range(40):
    optimizer.zero_grad(set_to_none=True)
    loss = bridge_loss()
    loss.backward()
    optimizer.step()
final_loss = bridge_loss().detach()
neighbors_after = nearest_ids(table.detach().clone())
REPORT = {
    "initial_loss": initial_loss.item(), "final_loss": final_loss.item(),
    "row4_neighbors_before": neighbors_before, "row4_neighbors_after": neighbors_after,
}

### Answer check

In [ ]:
assert initial_loss.item() > final_loss.item()
for row in (4, 5, 6, 7, 8, 9):
    assert not torch.allclose(table[row], before[row], atol=1e-6, rtol=1e-6)
unused_rows = (0, 1, 2, 3, 10, 11)
for row in unused_rows:
    assert torch.allclose(table[row], before[row], atol=1e-6, rtol=1e-6)
assert len(neighbors_before) == len(neighbors_after) == 3
assert 4 not in neighbors_before and 4 not in neighbors_after
assert set(context_ids.tolist()) == {4, 5, 6, 7, 8, 9}